<a href="https://www.kaggle.com/code/spyboy/tinotendanyashanu?scriptVersionId=151176407" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

The model acts like a sophisticated filter, trained to recognize and categorize comments based on their relevance to Physics, Chemistry, or Biology. It relies heavily on the quality and diversity of the training data (comments) it receives to understand the nuances of language and context associated with each scientific field. The model's effectiveness depends on its ability to accurately process, interpret, and classify the text data into the correct scientific category, making it a valuable tool for content management and analysis in educational or scientific online communities.

**This is the link to my colab** https://colab.research.google.com/drive/1CtDuDjak5vfUHkmJHyQgA0memOegeuDh?usp=sharing

**Link to kaggle**

https://www.kaggle.com/code/spyboy/tinotendanyashanu

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, AveragePooling1D, Flatten
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import models,layers


In [ ]:
import pandas as pd
from IPython.display import display
def display_all(df):
    with pd.option_context("display.max_rows", 3, "display.max_columns", 300,'display.max_colwidth', 3000):
        display(df)

In [ ]:
# Load the data
train_data = pd.read_csv('/kaggle/input/dataset/train.csv')
test_data = pd.read_csv('/kaggle/input/dataset/test.csv')


all_data = pd.concat([train_data, test_data])

all_data.head()


In [ ]:
# Cleaning and preprocesing the text data
import re

def clean_text(text):
    text = text.lower() 
    text = re.sub(r'\s+', ' ', text).strip()  
    text = re.sub(r'[^\w\s]', '', text)  
    return text

#  cleaning to the comments
train_data['Comment'] = train_data['Comment'].apply(clean_text)
test_data['Comment'] = test_data['Comment'].apply(clean_text)

#  missing values
train_data.dropna(subset=['Comment'], inplace=True)
test_data.dropna(subset=['Comment'], inplace=True)

In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(all_data['Comment'])
word_index = tokenizer.word_index

# Convert texts to sequences
train_sequences = tokenizer.texts_to_sequences(train_data['Comment'])
test_sequences = tokenizer.texts_to_sequences(test_data['Comment'])

# Padding
max_length = 100
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

# Label encoding
encoder = LabelEncoder()
encoder.fit(train_data['Topic'])
train_labels = encoder.transform(train_data['Topic'])
test_labels = encoder.transform(test_data['Topic'])

print (train_sequences[0])
print (train_padded[0])

In [ ]:
print (test_sequences[0])
print (test_padded[0])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(train_padded, train_labels, test_size=0.1, random_state=25)


In [ ]:
model = Sequential([
    Embedding(input_dim=10000, output_dim=32, input_length=max_length),
    Conv1D(130, 4, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(len(np.unique(train_labels)), activation='softmax')
])


model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


In [ ]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)


In [ ]:
test_loss, test_accuracy = model.evaluate(test_padded, test_labels)
print(f"Test accuracy: {test_accuracy}")